# Forge Knowledge Processing

Prepare downloaded documentation for chunking by cleaning, normalizing, and structuring the raw content.

In [23]:
!pip install trafilatura

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.6/134.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 21.7 MB/s eta 0:00:00


In [28]:
from pathlib import Path
import json
import re
import html

import pandas as pd
import trafilatura

from bs4 import BeautifulSoup
from google.colab import drive

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
PROJECT_ROOT = Path("/content/drive/MyDrive/forge")

CONFIG = {
    "project_root": PROJECT_ROOT,
    "sources": PROJECT_ROOT / "sources",
    "raw": PROJECT_ROOT / "knowledge_base" / "raw",
    "processed": PROJECT_ROOT / "knowledge_base" / "processed",
    "chunks": PROJECT_ROOT / "knowledge_base" / "chunks",
    "embeddings": PROJECT_ROOT / "knowledge_base" / "embeddings",
    "vector_store": PROJECT_ROOT / "knowledge_base" / "vector_store",
}

for path in [
    CONFIG["processed"],
    CONFIG["chunks"],
    CONFIG["embeddings"],
    CONFIG["vector_store"],
]:
    path.mkdir(parents=True, exist_ok=True)

#Verify Project Structure

In [4]:
print(f"Raw Documents Directory : {CONFIG['raw']}")
print(f"Processed Directory     : {CONFIG['processed']}")

print()

print("Raw Exists       :", CONFIG["raw"].exists())
print("Processed Exists :", CONFIG["processed"].exists())

Raw Documents Directory : /content/drive/MyDrive/forge/knowledge_base/raw
Processed Directory     : /content/drive/MyDrive/forge/knowledge_base/processed

Raw Exists       : True
Processed Exists : True


## Load Raw Documents

In [26]:
raw_documents = sorted(CONFIG["raw"].rglob("*.txt"))

print(f"Total Documents: {len(raw_documents)}")

Total Documents: 283


## Normalize Whitespace

In [16]:
def normalize_whitespace(text):
    text = text.replace("\r", "\n")
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)

    return text.strip()

##Extract Document

In [29]:
def extract_document(html_text):
    content = trafilatura.extract(html_text)

    if content:
        return normalize_whitespace(content)

    soup = BeautifulSoup(html_text, "html.parser")

    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    text = html.unescape(text)

    return normalize_whitespace(text)

In [27]:
sample_document = raw_documents[0]

with open(sample_document, "r", encoding="utf-8") as file:
    html_text = file.read()

processed_text = extract_document(html_text)

print("File:", sample_document.relative_to(CONFIG["raw"]))
print()
print(processed_text[:1000])

File: anthropic_claude/api_reference.txt

The Claude API is a RESTful API at https://api.anthropic.com that provides programmatic access to Claude models and Claude Managed Agents.
New to Claude? For direct model access, start with Get started and Working with Messages. For managed agent infrastructure, see the Claude Managed Agents quickstart.
To use the Claude API, you'll need:
For step-by-step setup instructions, see Get started.
The Claude API includes the following APIs:
General Availability:
POST /v1/messages)POST /v1/messages/batches)POST /v1/messages/count_tokens)GET /v1/models)Beta:
POST /v1/files, GET /v1/files)POST /v1/skills, GET /v1/skills)POST /v1/agents, GET /v1/agents)POST /v1/sessions, GET /v1/sessions/{id}/stream)POST /v1/environments, GET /v1/environments)For the complete API reference with all endpoints, parameters, and response schemas, explore the API reference pages listed in the navigation. To access beta features, see Beta headers.
For details on both authentic

## Process Documents

In [30]:
def process_document(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        raw_html = file.read()

    processed_text = extract_document(raw_html)

    return {
        "technology": file_path.parent.name,
        "source": file_path.stem,
        "path": str(file_path.relative_to(CONFIG["raw"])),
        "content": processed_text,
        "character_count": len(processed_text),
    }

## Process All Documents

In [31]:
processed_documents = []

for file_path in raw_documents:
    try:
        document = process_document(file_path)
        processed_documents.append(document)
    except Exception as error:
        print(f"Failed: {file_path.name}")
        print(error)

print(f"Successfully Processed : {len(processed_documents)}")

ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None


Successfully Processed : 283


## Inspect Processed Document

In [32]:
sample = processed_documents[0]

print("Technology      :", sample["technology"])
print("Source          :", sample["source"])
print("Characters      :", sample["character_count"])

print("\n")

print(sample["content"][:2000])

Technology      : anthropic_claude
Source          : api_reference
Characters      : 7202


The Claude API is a RESTful API at https://api.anthropic.com that provides programmatic access to Claude models and Claude Managed Agents.
New to Claude? For direct model access, start with Get started and Working with Messages. For managed agent infrastructure, see the Claude Managed Agents quickstart.
To use the Claude API, you'll need:
For step-by-step setup instructions, see Get started.
The Claude API includes the following APIs:
General Availability:
POST /v1/messages)POST /v1/messages/batches)POST /v1/messages/count_tokens)GET /v1/models)Beta:
POST /v1/files, GET /v1/files)POST /v1/skills, GET /v1/skills)POST /v1/agents, GET /v1/agents)POST /v1/sessions, GET /v1/sessions/{id}/stream)POST /v1/environments, GET /v1/environments)For the complete API reference with all endpoints, parameters, and response schemas, explore the API reference pages listed in the navigation. To access beta feature

## Save Processed Documents

In [33]:
CONFIG["processed"].mkdir(parents=True, exist_ok=True)

for document in processed_documents:
    output_dir = CONFIG["processed"] / document["technology"]
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / f"{document['source']}.json"

    with open(output_file, "w", encoding="utf-8") as file:
        json.dump(document, file, indent=2, ensure_ascii=False)

print(f"Saved {len(processed_documents)} processed documents.")

Saved 283 processed documents.


## Generate Processing Report

In [34]:
report = pd.DataFrame([
    {
        "technology": document["technology"],
        "source": document["source"],
        "characters": document["character_count"],
        "path": document["path"],
    }
    for document in processed_documents
])

report_path = CONFIG["project_root"] / "knowledge_base" / "processing_report.csv"

report.to_csv(report_path, index=False)

report.head()

,technology,source,characters,path
0,anthropic_claude,api_reference,7202,anthropic_claude/api_reference.txt
1,anthropic_claude,github_repository,613,anthropic_claude/github_repository.txt
2,anthropic_claude,official_documentation,1087,anthropic_claude/official_documentation.txt
3,anthropic_claude,release_notes,50036,anthropic_claude/release_notes.txt
4,anthropic_claude,technical_blog,1291,anthropic_claude/technical_blog.txt


## Processing Summary

In [35]:
print("=" * 50)
print("DOCUMENT PROCESSING SUMMARY")
print("=" * 50)

print(f"Raw Documents        : {len(raw_documents)}")
print(f"Processed Documents  : {len(processed_documents)}")
print(f"Output Directory     : {CONFIG['processed']}")
print(f"Processing Report    : {report_path}")

total_characters = report["characters"].sum()
average_characters = report["characters"].mean()

print(f"Total Characters     : {total_characters:,}")
print(f"Average Characters   : {average_characters:,.0f}")

print("\nDocuments per Technology:")
print(report["technology"].value_counts().sort_index())

DOCUMENT PROCESSING SUMMARY
Raw Documents        : 283
Processed Documents  : 283
Output Directory     : /content/drive/MyDrive/forge/knowledge_base/processed
Processing Report    : /content/drive/MyDrive/forge/knowledge_base/processing_report.csv
Total Characters     : 2,942,579
Average Characters   : 10,398

Documents per Technology:
technology
anthropic_claude       5
arize_ai               1
arize_phoenix          2
aws_sagemaker          1
axolotl                2
                      ..
vertex_ai              1
vllm                   2
weaviate               7
weights_biases         2
zero_shot_prompting    1
Name: count, Length: 114, dtype: int64
